# Compute fitness effects for subset trees

Use expected rates from the global neutral model to compute fitness effects for each subset (host, geographic, split-half).

In [1]:
import pandas as pd
import numpy as np

## Read data

In [2]:
# Read subset counts
counts_df = pd.read_csv('../results/subset_counts.csv', keep_default_na=False, low_memory=False)
counts_df = counts_df.replace('', np.nan)
counts_df['codon_site'] = counts_df['codon_site'].astype(str)
print(f'Subset counts: {len(counts_df):,} rows')
print(f'Subsets: {sorted(counts_df["subset"].unique())}')

Subset counts: 4,361,835 rows
Subsets: ['asia', 'avian', 'europe', 'human', 'north_america', 'split_a', 'split_b', 'swine']


In [3]:
# Read expected rates from the global neutral model
predicted_rates_df = pd.read_csv('../results/expected_rates.csv', keep_default_na=False)
del predicted_rates_df['tau_squared']
predicted_rates_df.head()

,mut_type,segment,motif,predicted_rate
0,AC,HA,AAA,0.200626
1,AC,HA,AAC,0.319609
2,AC,HA,AAG,0.248512
3,AC,HA,AAT,0.313680
4,AC,HA,CAA,0.177674


## Compute expected counts and filter

In [4]:
counts_cutoff = 1

actual_expected_df = (
    counts_df
    .merge(predicted_rates_df, on=['mut_type', 'segment', 'motif'], how='left')
    .assign(expected_count=lambda x: x['predicted_rate'] * x['evo_opp'])
    .query('actual_count >= @counts_cutoff | expected_count >= @counts_cutoff')
)
print(f'Rows after filtering (>= {counts_cutoff} actual or expected): {len(actual_expected_df):,}')
actual_expected_df.head()

Rows after filtering (>= 1 actual or expected): 873,619


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,subtype,segment,segment_subtype,segment_length,subset,subset_type,evo_opp,rate,predicted_rate,expected_count
3,1577,A1577C,A,C,PB2,2,526,AAG,ACG,K,...,all,PB2,PB2_all,2277,split_b,split_half,19.101010,0.0523532522474881,0.236733,4.521845
4,1577,A1577C,A,C,PB2,2,526,AAA,ACA,K,...,all,PB2,PB2_all,2277,split_b,split_half,7.574440,0.0,0.191117,1.447605
6,72,A72C,A,C,NP,3,24,GAA,GAC,E,...,all,NP,NP_all,1494,europe,geographic,11.448461,0.0,0.215537,2.467567
7,71,A71C,A,C,NP,2,24,GAG,GCG,E,...,all,NP,NP_all,1494,europe,geographic,12.514056,0.0,0.116510,1.458018
8,71,A71C,A,C,NP,2,24,GAA,GCA,E,...,all,NP,NP_all,1494,europe,geographic,11.449130,0.0,0.094060,1.076905


## Compute amino-acid-level fitness effects

In [5]:
pseudo_count = 0.5

groupby_cols = [
    'subset', 'subset_type', 'subtype', 'segment', 'gene',
    'codon_site', 'wt_aa', 'mut_aa', 'aa_mut', 'mut_class'
]

aa_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x:
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)

aa_fitness_df.to_csv('../results/subset_aa_fitness_effects.csv', index=False)
print(f'AA-level fitness effects: {len(aa_fitness_df):,} rows')
aa_fitness_df.head()

AA-level fitness effects: 458,572 rows


,subset,subset_type,subtype,segment,gene,codon_site,wt_aa,mut_aa,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,asia,geographic,H1,HA,HA,1,M,I,M1I,nonsynonymous,0,38.077221,-4.345809
1,asia,geographic,H1,HA,HA,1,M,K,M1K,nonsynonymous,0,4.618749,-2.326057
2,asia,geographic,H1,HA,HA,1,M,R,M1R,nonsynonymous,0,2.930134,-1.925746
3,asia,geographic,H1,HA,HA,1,M,T,M1T,nonsynonymous,0,13.860795,-3.357649
4,asia,geographic,H1,HA,HA,10,C,S,C10S,nonsynonymous,1,0.085551,0.940668


In [6]:
# Summary by subset and mutation class
aa_fitness_df.groupby(['subset', 'mut_class']).size().unstack(fill_value=0)

mut_class,nonsense,nonsynonymous,synonymous
subset,,,
asia,3014,51359,10202
avian,2559,45410,9726
europe,2528,41274,8223
human,2391,40619,7315
north_america,2762,47249,9445
split_a,3060,55066,10876
split_b,3068,55017,10929
swine,1643,28353,6484
